In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import json
import numpy
from datasets import load_dataset
import tqdm

index = faiss.read_index("my_rag_db.index")
with open("my_rag_db.json", "r") as f:
    metadata = json.load(f)

embedder = SentenceTransformer("all-MiniLM-L6-v2")

dataset = load_dataset("natural_questions", split="train", streaming=True)



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1694.88it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [27]:
eval_set = []

NB_QUESTIONS_TEST = 200

for i, example in enumerate(tqdm.tqdm(dataset, total=20000)):
    question_text = example["question"]["text"]
    url_doc = example["document"]["url"]
    
    eval_set.append({
        "question": question_text,
        "ground_truth_url": url_doc
    })
    
    if len(eval_set) > NB_QUESTIONS_TEST:
        break
    

  1%|          | 200/20000 [00:07<12:50, 25.70it/s] 


In [28]:
import numpy as np

all_relevance_results = []
K_TOP = 20

for question in eval_set:
    embedding = embedder.encode([question["question"]], convert_to_numpy=True)
    faiss.normalize_L2(embedding)
    distances, indices = index.search(embedding, K_TOP)
    
    current_query_relevance = []
    
    for idx in indices[0]:
        found_url = metadata[idx]["url"]
        is_relevant = (found_url == question["ground_truth_url"])
        current_query_relevance.append(is_relevant)
    
    all_relevance_results.append(current_query_relevance)

all_relevance_results[0]
    


[True,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 True,
 False,
 False,
 False,
 False,
 False,
 False]

In [29]:
def calculate_ap(relevance_bools):
    precisions = []
    num_relevant_found = 0
    
    for i, is_relevant in enumerate(relevance_bools):
        rank = i+1
        
        if is_relevant:
            num_relevant_found+=1
            precision_at_rank = num_relevant_found/rank
            precisions.append(precision_at_rank)
    
    if not precisions:
        return 0.0
    
    return sum(precisions)/len(precisions)

ap_scores = [calculate_ap(res) for res in all_relevance_results]
map_score = sum(ap_scores)/len(ap_scores)

print(f"Nombre de questions: {len(ap_scores)}")
print(f"MAP@{K_TOP} : {map_score:.2f}")


Nombre de questions: 201
MAP@20 : 0.58
